# 02. 유익한 환경 관찰 선택하기

## 목표
모든 도구 출력이 세계 모델 학습에 유익하지 않다는 점을 코드로 확인합니다. 반복 오류, 행동의 단순 반향, 무변화 관찰을 필터링합니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Transition:
    action: str
    observation: str
    changed_state: bool

transitions = [
    Transition("mkdir output", "created directory output", True),
    Transition("bad-command", "command not found", False),
    Transition("bad-command", "command not found", False),
    Transition("echo hello", "hello", False),
    Transition("pytest", "2 passed in 0.4s", True),
    Transition("open docs", "open docs", False),
]

In [ ]:
def select_informative(items):
    selected = []
    seen_failures = set()
    for item in items:
        normalized_action = item.action.strip().lower()
        normalized_observation = item.observation.strip().lower()
        if normalized_action == normalized_observation:
            continue  # 행동을 그대로 복사한 관찰은 새 정보가 거의 없습니다.
        is_error = "not found" in normalized_observation or "error" in normalized_observation
        if is_error:
            signature = (normalized_action, normalized_observation)
            if signature in seen_failures:
                continue  # 같은 실패를 반복 학습해 오류 패턴에 과적합하지 않게 합니다.
            seen_failures.add(signature)
        if not item.changed_state and not is_error:
            continue
        selected.append(item)
    return selected

selected = select_informative(transitions)
for item in selected:
    print(f"KEEP | {item.action!r} -> {item.observation!r}")
print(f"선택 비율: {len(selected)}/{len(transitions)}")

## 확장 과제

- 관찰 길이, action entropy, 상태 diff 크기를 이용해 0~1 유익성 점수를 만드세요.
- 문서 검색 출력과 코드 실행 출력을 구분하고 서로 다른 keep ratio를 적용하세요.
- 필터 전후 데이터에서 도구별 비중이 지나치게 달라지는지 표로 확인하세요.
- 실패를 모두 제거하지 마세요. 첫 실패 관찰은 복구 행동을 배우는 중요한 신호일 수 있습니다.